### 1. Scaled dot product Attention

In [1]:
# 예시 문장
sentences = [
    'I love Deep learning',
    'I love Pytorch',
    'Attention is powerful',
    'deep learnig is fun'
]

# 토큰화 vocabulary 만들기 (공백 기준)
tokenized_sentences = [s.split() for s in sentences]

# vocabulary 생성
special_tokens = ["<PAD>", "<UNK>"]

# 중복 단어 수집
word_set = set()
for tokens in tokenized_sentences:
    word_set.update(tokens)

vocab = special_tokens + sorted(list(word_set))
word2idx = {word:idx for idx, word in enumerate(vocab)}
idx2word = {idx:word for word, idx in word2idx.items()}

print("vocab", vocab)
print("vocab size", len(vocab))
print("word2idx", word2idx)

# 문장을 인덱스로 변환 
## 문장 길이가 다르기 때문에 padding을 함.

max_len = max(len(tokens) for token in tokenized_sentences)
# 데이터셋 내에서 가장 긴 문장의 길이를 구함
print("max_len", max_len)

def encode_sentence(tokens, word2idx, max_len):
    ids = [word2idx.get(token, word2idx['<UNK>']) for token in tokens]
    # 단어를 사전에 정의된 숫자로 바꾸되, 없는 단어는 <UNK>로 처리

    #padding
    ids += [word2idx["<PAD>"]] * (max_len - len(ids))
    # 모든 문장의 길이를 맞추기 위해 부족한 공간은 <PAD>로 처리
    return ids

encoded_sentences = [encode_sentence(tokens, word2idx, max_len) for tokens in tokenized_sentences]
# 모든 문장 리스트에 대해 위 인코딩 과정을 적용

for s,enc in zip(sentences, encoded_sentences):
    print(f"{s:30} -> {enc}")


vocab ['<PAD>', '<UNK>', 'Attention', 'Deep', 'I', 'Pytorch', 'deep', 'fun', 'is', 'learnig', 'learning', 'love', 'powerful']
vocab size 13
word2idx {'<PAD>': 0, '<UNK>': 1, 'Attention': 2, 'Deep': 3, 'I': 4, 'Pytorch': 5, 'deep': 6, 'fun': 7, 'is': 8, 'learnig': 9, 'learning': 10, 'love': 11, 'powerful': 12}
max_len 4
I love Deep learning           -> [4, 11, 3, 10]
I love Pytorch                 -> [4, 11, 5, 0]
Attention is powerful          -> [2, 8, 12, 0]
deep learnig is fun            -> [6, 9, 8, 7]


In [2]:
# 텐서로 만들기
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_printoptions(precision=4,sci_mode=False)

input_ids = torch.tensor(encoded_sentences) # (batch_size(문장 개수), seq_len(=padding 포함 최대 길이)
print("input_ids shape:", input_ids.shape)
print(input_ids)

input_ids shape: torch.Size([4, 4])
tensor([[ 4, 11,  3, 10],
        [ 4, 11,  5,  0],
        [ 2,  8, 12,  0],
        [ 6,  9,  8,  7]])


In [3]:
# Embedding 만들기
vocab_size = len(vocab)
embed_dim = 8

embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim = embed_dim)

embedded = embedding(input_ids) #(batch_size, seq_len, embed_dim)

print("embedd shape:", embedded.shape)
print(embedded[0])

embedd shape: torch.Size([4, 4, 8])
tensor([[-0.1146, -1.8382,  1.0399, -0.0818,  0.3315, -1.9526, -2.5461,  0.0502],
        [-0.1326,  0.4343, -0.9279, -0.6326,  0.0791, -1.1689,  1.3920, -1.2375],
        [ 0.4997, -0.1777,  1.2278,  0.8079, -0.7225,  1.1114,  0.1896,  0.2296],
        [-0.4301,  0.4847,  0.6238,  0.2534, -1.4638, -0.2281, -1.0945,  1.5567]],
       grad_fn=<SelectBackward0>)


In [4]:
# Q, K, V
d_k = 8
d_v = 8

W_q = nn.Linear(embed_dim, d_k, bias=False)
W_k = nn.Linear(embed_dim, d_k, bias=False)
W_v = nn.Linear(embed_dim, d_v, bias=False)

Q = W_q(embedded) #(batch, seq_len, d_k)
K = W_k(embedded) #(batch, seq_len, d_k)
V = W_v(embedded) #(batch, seq_len, d_v)

print("Q shpae:", Q.shape)
print("K shpae:", K.shape)
print("V shpae:", V.shape)

Q shpae: torch.Size([4, 4, 8])
K shpae: torch.Size([4, 4, 8])
V shpae: torch.Size([4, 4, 8])


In [5]:
# Padding Mask 만들기
# <PAD>는 실제 의미 있는 단어가 아님. -> Attention에서는 무시

pad_idx = word2idx["<PAD>"]

# 실제 토큰이면 1, PAD이면 0
attention_mask = (input_ids !=pad_idx).int()
print("attention_mask shape:", attention_mask.shape)
print(attention_mask)

# attention score에 적용할 수 있겍  shape을 바꿔줌.
# score shape: (batch, seq_len, seq_len)
# mask broadcast 가능하게 (batch,1,seq_len)

key_padding_mask = attention_mask.unsqueeze(1) # (batch,1,seq_len)
print("key_padding_mask shape:", key_padding_mask.shape)

attention_mask shape: torch.Size([4, 4])
tensor([[1, 1, 1, 1],
        [1, 1, 1, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
key_padding_mask shape: torch.Size([4, 1, 4])


In [6]:
# Scaled dot-product Attention 함수
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q: (batch, seq_len_q, d_k)
    # K: (batch, seq_len_k, d_k)
    # V: (batch, seq_len_k, d_v)
    # mask: (batch,1,seq_len_k)

    d_k = Q.size(-1)
    # 스케일링을 위해 key(K) 벡터의 차원 수(d_k) 추출

    # (batch. seq_len_q, seq_len_k)
    scores = torch.matmul(Q, K.transpose(-2,-1)) / (d_k **0.05)
    # Q와 K의 전치행렬(transpose)을 곱한 후 루트 d_k로 나누어 유사도 점수(score)를 계산

    if mask is not None:
        scores = scores.masked_fill(mask==0, float('-inf'))
    # 마스크 값이 0인 위치(패딩 등)를 매우 작은 음수(-inf)로 채워 어텐션에서 제외

    attn_weights = torch.softmax(scores,dim=-1)
    # softmax를 적용하여 각 단어 간의 관계를 0~1 사이의 확률값(가중치)로 변환

    output = torch.matmul(attn_weights, V)
    # 가중치와 값(V) 행렬을 곱하여 최종 attention 값(context Vector)을 산출
    
    return output,attn_weights, scores
    
# attention 계산 실행
output, attn_weights, scores = scaled_dot_product_attention(Q, K, V, key_padding_mask)

print("scores shape:", scores.shape)                # scores: (batch, seq_len, seq_len)
print("attn_weight shape:", attn_weights.shape)     # attn_weights: (batch, seq_len, seq_len)
print("output shape:", output.shape)                # output:(batch,seq_len,d_k)


scores shape: torch.Size([4, 4, 4])
attn_weight shape: torch.Size([4, 4, 4])
output shape: torch.Size([4, 4, 8])


In [7]:
# 첫 번째 문장 attention 문장
## 'I love Deep learning',

sample_idx = 0
print("sentence:", sentences[sample_idx])
print("token ids:", input_ids[sample_idx])

tokens = [idx2word[idx.item()] for idx in input_ids[sample_idx]]
print("tokens:", tokens)

print("attention weight for sentence:0")
print(attn_weights[sample_idx])

sentence: I love Deep learning
token ids: tensor([ 4, 11,  3, 10])
tokens: ['I', 'love', 'Deep', 'learning']
attention weight for sentence:0
tensor([[0.6305, 0.1054, 0.0642, 0.1998],
        [0.3304, 0.1647, 0.4070, 0.0979],
        [0.0863, 0.4373, 0.2844, 0.1919],
        [0.0456, 0.1100, 0.1716, 0.6728]], grad_fn=<SelectBackward0>)


In [8]:
# 보기 좋게 출력하는 함수
def print_attention_matrix(attn_weights, tokens):
    print("f{'':.12}", end="")
    for token in tokens:
        print(f"{token:>12}", end="")
    print()

    for i,row_token in enumerate(tokens):
        print(f"{row_token:12}", end="")
        for j in range(len(tokens)):
            print(f"{attn_weights[i,j].item():12.4f}",end="")
        print()

print_attention_matrix(attn_weights[sample_idx], tokens)

f{'':.12}           I        love        Deep    learning
I                 0.6305      0.1054      0.0642      0.1998
love              0.3304      0.1647      0.4070      0.0979
Deep              0.0863      0.4373      0.2844      0.1919
learning          0.0456      0.1100      0.1716      0.6728


In [9]:
# padding이 있는 문장 확인
sample_idx = 1

print("sentence:", sentences[sample_idx])
tokens = [idx2word[idx.item()] for idx in input_ids[sample_idx]]
print("tokens:", tokens)

print_attention_matrix(attn_weights[sample_idx], tokens)

sentence: I love Pytorch
tokens: ['I', 'love', 'Pytorch', '<PAD>']
f{'':.12}           I        love     Pytorch       <PAD>
I                 0.7217      0.1207      0.1577      0.0000
love              0.4082      0.2035      0.3884      0.0000
Pytorch           0.1849      0.3892      0.4259      0.0000
<PAD>             0.6813      0.1362      0.1825      0.0000


### 2. Self-Attention


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

torch.manual_seed(42)
torch.set_printoptions(precision=4, sci_mode=False)

# 데이터
data = [
    ("i love the movie",1),
    ("this film is great",1),
    ("i really like this",1),
    ("this is amazing",1),
    ("i hate this movie",0),
    ("this film is terrible",0),
    ("this is boring",0)
]
#지도학습 -> 분류-> 라벨이 잘못 달려있으면 이상하게 학습함.

[('i love the movie', 1), ('this film is great', 1), ('i really like this', 1), ('this is amazing', 1), ('i hate this movie', 0), ('this film is terrible', 0), ('this is boring', 0)]


In [ ]:
# 토큰화 및 vocabulary 생성
def tokenize(text):
    return text.lower().split()
# 텍스트를 소문자로 바꾸고 공백 기준으로 나누어 단어 리스트(토큰)으로 만들.ㅁ

special_tokens = ["<PAD>", "<UNK>"]
# 문장 길이를 맞추기 위한 <PAD>, 모르는 단어 처리를 위한 <UNK> 특수 토큰을 정의

word_set = set()
for sentence, label in data:
    word_set.update(tokenize(sentence))
# 중복을 허용하지 않는 집합(set)을 생성하여 전체 데이터의 고유 단어를 수집.

vocab = special_tokens + sorted(list(word_set))
# 특수 토큰과 알파벳순로 정렬된 단어들을 합쳐 최종 단어 사전(vocab)을 만듦.
word2idx = {word:idx for idx, word in enumerate(vocab)}
# 단어를 넣으면 숫자로 나오는 "단어-번호" 딕셔너리를 만듦(인코더용)
idx2word = {idx:word for word, idx in word2idx.items()}
# 숫자를 넣으면 단어가 나오는 "번호-단어" 딕셔너리를 만듦(디코더용)

pad_idx = word2idx["<PAD>"]
unk_idx = word2idx["<UNK>"]
#나중에 패딩 처리에 사용할 <PAD>와 <UNK>의 고유 번호를 변수에 저장

print("vocab:", vocab)
print("vocab size:", len(vocab))

vocab: ['<PAD>', '<UNK>', 'amazing', 'boring', 'film', 'great', 'hate', 'i', 'is', 'like', 'love', 'movie', 'really', 'terrible', 'the', 'this']
vocab size: 16


In [ ]:
# 문장을 인덱스로 반환
max_len = max(len(tokenize(sentence)) for sentence, _ in data)
# 모든 문장 중 가장 긴 뭊앙의 단어 개수를 찾아 기준(max_len)

def encode(sentence):
    tokens = tokenize(sentence)
    # 입력받은 문장을 단위 단위로 쪼갬(토큰화)
    ids = [word2idx.get(token, unk_idx) for token in tokens]
    # 단어를 사전에 등록된 숫자로 바꿈. 사전에 없으면 <UNK>로 처리
    ids += [pad_idx] * (max_len - len(ids))
    # 문장 길이를 max_len를 맞추기 위해 부족한 뒷부분을 <PAD> 번호로 부여
    return ids

encoded_data = [(encode(sentence), label) for sentence, label in data]
# 전체 데이터 (문장,라벨)를 돌면서 문장을 숫자 리스트로 변환하여 저장

print("Encoded Samples:")
for i in range(len(encoded_data)):
    print(data[i][0], "->", encoded_data[i][0], "label:",encoded_data[i][1])
# 원본 문장과 변환된 숫자(인덱스) 리스트, 그리고 정답(라벨)을 출력하여 확인.

Encoded Samples:
i love the movie -> [7, 10, 14, 11] label: 1
this film is great -> [15, 4, 8, 5] label: 1
i really like this -> [7, 12, 9, 15] label: 1
this is amazing -> [15, 8, 2, 0] label: 1
i hate this movie -> [7, 6, 15, 11] label: 0
this film is terrible -> [15, 4, 8, 13] label: 0
this is boring -> [15, 8, 3, 0] label: 0


In [ ]:
# Dataset / DataLoader
class TextDataset(Dataset):
    # 파이토치 전용 Dataset 클래스를 정의
    def __init__(self,encode_data):
        self.encoded_data = encoded_data
        #초기화 단계: 인코딩된 데이터를 클래스 내부 변수에 저장
    def __len__(self):
        return len(self.encoded_data)
        # 데이터셋 전체의 개수를 반환(len(dataset) 호출시 실행
    def __getitem__(self,idx):
        # __getitem__
        ## 슬라이싱을 구현할 수 있도록 도우며 리스트에서 슬라이싱을 하게 되면 내부적으로 메서드를 실행함.
        
        input_ids, label = self.encoded_data[idx]
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)
        # 특정 인덱스(idx)의 데이터를 가져와 파이토치 텐서 형태로 반환
dataset = TextDataset(encoded_data)
# 위에서 만든 클래스를 이용해 실제 데이터셋 객체를 생성
loader = DataLoader(dataset, batch_size=4, shuffle=True)
#데이터 로더: 데이터를 4개씩 묶고(batch_size), 순서를 섞어서(shuffle)

In [ ]:
class Dog:
    # 1. 클래스 변수(전역변수, 모든 강아지가 공유)
    speices = "Mammal"

    def __init__(self, name,age):
        # 2. 인스턴스 변수(각 강아지마다 개별적으로 가지는 값)
        self.name = name
        self.age = age
    def introduce(self,greeting):
        # 3. 로컬변수(이 함수안에서만 잠깐 쓰임)
        full_greeting = f"{greeting}! I am {self.name}"

        # 클래스 변수는 "클래스이름.변수"로 접근하는 게 정석
        print(full_greeting)
        print(f"speices: {Dog.speices}")

## 테스트
dog1 = Dog("Happy",3)
dog2 = Dog("Lucky",5)

#인스턴스 변수 확인:각자 다름
print(dog1.name)
print(dog2.name)

# 클래스 변수 확인: 모두 같음
print(dog1.speices)
print(dog2.speices)

# 클래스 변수 변경: 모든 객체에 영향을 줌
Dog.speices = "Canine"
print(dog1.speices)

$$
Attenion Scores = softmax( {\dfrac {Q \cdot K^T}{\sqrt{d_k}}})
$$

In [ ]:
# Scaled dot-product Attention 함수
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q: (batch, seq_len_q, d_k)
    # K: (batch, seq_len_k, d_k)
    # V: (batch, seq_len_v, d_v)
    # mask: (batch,1,seq_len_k)

    d_k = Q.size(-1)
    # 스케일링을 위해 key(K) 벡터의 차원 수(d_k) 추출

    # (batch. seq_len_q, seq_len_k)
    scores = torch.matmul(Q, K.transpose(-2,-1)) / (d_k **0.5)
    # Q와 K를 내적(matmul) 한 뒤 루트 d_k로 나누어 유사도 점수를 계산
    # K.transpose(-2,-1): (batch,d_k,seq_len_k)
    # Q * K^T -> n x d_k * (d_k x n) -> nxn

    if mask is not None:
        scores = scores.masked_fill(mask==0, float('-inf'))
    # 마스크가 있다면, 패딩위치(0) 매우 작은 음수(-inf)로 채워 어텐션에서 제외

    attn_weights = torch.softmax(scores,dim=-1)
    # softmax를 적용하여 각 단어 간의 관계를 0~1 사이의 확률값(가중치)로 변환

    output = torch.matmul(attn_weights, V)
    # 가중치와 값(V) 행렬을 곱하여 최종 attention 값(context Vector)을 산출
    
    return output, attn_weights

In [ ]:
# Self-Attention Classifier
class SelfAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx = pad_idx)
        # 단어 번호를 고정된 크기의 연속적인 벡터(임베딩)로 변환하는 층.
        self.W_q = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, hidden_dim, bias=False)
        # 입력 벡터(x)로 부터 Query,key,value를 추출하기 위한 선형 변환 층.
        self.classifier = nn.Linear(hidden_dim, num_classes)
        # 최종 추출된 문장 특징 벡터를 클래스 개수(ex.2개)만큼의 점수(Logits)로 변환
    def forward(self,input_ids):
        
        # 1)padding_mask
        mask = (input_ids != self.pad_idx).unsqueeze(1) # (batch,1,seq_len)
        # 패딩 토큰(<PAD>) 위치를 0으로 마스킹하여 어텐션 연산 시 제외하도록 설정

        # embedding
        x = self.embedding(input_ids) #(batch, seq_len, embed_dim)
        # 정수 형태의 단어 ID들을 임베딩 벡터로 변환.

        # Q,K,V
        Q = self.W_q(x) #(batch,seq_len, hidden_dim)
        K = self.W_k(x)
        V = self.W_v(x)
        # 임베딩된 벡터에 각각 가중치를 곱해 Q, K, V 행렬을 생성

        # self-attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V,mask)
        # 단어들끼리 서로 얼마나 관련이 있는지 계산하여 문맥이 반영된 벡터(attn_vector)

        # masked mean pooling
        valid_mask = (input_ids != self.pad_idx).unsqueeze(-1) #(batch, seq_len,1)
        # 패딩 토큰 위치는 0(False), 실제 단위 위치는 1(True)로 표시한 마스크를 만듦.
        attn_output = attn_output * valid_mask
        # 어텐션 결과물에서 패딩 부분의 값들을 0으로 지워 계산에서 제외

        sum_output = attn_output.sum(dim=1) # (batch,hidden)
        # 패딩을 제외한 실제 단어의 벡터를 모두 더함

        valid_token_count = valid_mask.sum(dim=1).clamp(min=1) #(batch,1)
        # 각 문장별로 패딩이 아닌 실제 단어가 몇 개인지 세어줌
        # (0으로 나누기 방지를 위해 최소값 1을 설정

        pooled = sum_output / valid_token_count
        # 총합을 실제 단어 개수로 나누어, 순수하게 단어 정보만 반영된 평균 벡터를 얻음

        # Classifier
        logits = self.classifier(pooled) # (batch, num_classes)
        # 딥러닝 분류 모델의 가장 마지막 층에서 출력되는 softmax함수를 거치기 전의 가공되지 않은 예측값
        # softmax(로짓 점수)-> 모든 값이 합이 1인 확률로 변함
        return logits, attn_weights


In [16]:
# 모델 생성
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model = SelfAttentionClassifier(
    vocab_size=len(vocab),
    embed_dim=16,
    hidden_dim=16,
    num_classes=2,
    pad_idx=pad_idx
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 30

for epoch in range(epochs):
    model.train() #모델을 훈련 모드로 설정(dropout, batch normalization 등이 활성화)
    total_loss = 0
    correct = 0
    total = 0

    for input_ids, labels in loader:
        # 데이터 로드에서 배치 단위(4개씩)로 데이터를 꺼내옴
        input_ids = input_ids.to(device)
        # 데이터를 GPU 혹은 CPU장치로 옮김.
        labels = labels.to(device)

        optimizer.zero_grad()
        # 이전 단계에서 계산된 기울기를 0으로 초기화

        logits, attn_weights = model(input_ids)
        # 모델에 데이터를 넣어 예측값(logits)과 어텐션 가중치를 얻음(순전파)
        loss = criterion(logits, labels)


        loss.backward() #오차를 바탕으로 역전파를 수행하며 각 파라미터의 기울기 계산
        optimizer.step() # 계산된 기울기를 바탕으로 모델의 가중치를 업데이트

        total_loss += loss.item() # 현재 배치의 오차 값을 누적

        preds = torch.argmax(logits, dim=1) #로짓 점수 중에 가장 높은 인덱스를 선택해 예측 클래스를 결정
        correct += (preds == labels).sum().item()
        # 예측이 맞은 개수를 누적하고 전체 데이터 개수도 업데이트
        total += labels.size(0)

    acc = correct / total
    print(f"Epoch {epoch+1:02d} | Loss: {total_loss:.4f} | Acc: {acc:.4f}")

Epoch 01 | Loss: 1.5197 | Acc: 0.2857
Epoch 02 | Loss: 1.4929 | Acc: 0.2857
Epoch 03 | Loss: 1.4540 | Acc: 0.2857
Epoch 04 | Loss: 1.4428 | Acc: 0.2857
Epoch 05 | Loss: 1.4468 | Acc: 0.2857
Epoch 06 | Loss: 1.4095 | Acc: 0.2857
Epoch 07 | Loss: 1.3982 | Acc: 0.4286
Epoch 08 | Loss: 1.4060 | Acc: 0.4286
Epoch 09 | Loss: 1.3529 | Acc: 0.4286
Epoch 10 | Loss: 1.3615 | Acc: 0.5714
Epoch 11 | Loss: 1.3653 | Acc: 0.7143
Epoch 12 | Loss: 1.3539 | Acc: 0.7143
Epoch 13 | Loss: 1.2933 | Acc: 0.7143
Epoch 14 | Loss: 1.3048 | Acc: 0.7143
Epoch 15 | Loss: 1.3009 | Acc: 0.7143
Epoch 16 | Loss: 1.2698 | Acc: 0.7143
Epoch 17 | Loss: 1.2684 | Acc: 0.7143
Epoch 18 | Loss: 1.2696 | Acc: 0.7143
Epoch 19 | Loss: 1.2185 | Acc: 0.7143
Epoch 20 | Loss: 1.1973 | Acc: 0.7143
Epoch 21 | Loss: 1.1795 | Acc: 0.7143
Epoch 22 | Loss: 1.1991 | Acc: 0.7143
Epoch 23 | Loss: 1.1936 | Acc: 0.7143
Epoch 24 | Loss: 1.1481 | Acc: 0.7143
Epoch 25 | Loss: 1.1586 | Acc: 0.7143
Epoch 26 | Loss: 1.1197 | Acc: 0.7143
Epoch 27 | L

In [18]:
# 예측 확인
def predict_sentence(model, sentence):
    model.eval()
    ids = encode(sentence)
    input_tensor = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():
        logits, attn_weights = model(input_tensor)
        pred = torch.argmax(logits,dim=1).item()

        tokens = [idx2word[idx] for idx in ids]
        return pred,attn_weights[0].cpu(), tokens
test_sentences = [
    "i love this",
    "this is terrible",
    "i like this movie",
    "this is boring"
]

print("Prediction result")
for sentence in test_sentences:
    pred, attn, tokens = predict_sentence(model, sentence)
    print(f"sentence: {sentence}")
    print("Predicted label:", pred, "(1=postive, 0=negative)")
    print("Tokens:", tokens)
    print("-"*50)

sentences = ["this is terrible"]
pred,  attn, tokens = predict_sentence(model, sentence)

print("Attention matrix example")
print("Sentence:", sentences)
print("Predicted Label:",pred)
print_attention_matrix(attn,tokens)

Prediction result
sentence: i love this
Predicted label: 0 (1=postive, 0=negative)
Tokens: ['i', 'love', 'this', '<PAD>']
--------------------------------------------------
sentence: this is terrible
Predicted label: 0 (1=postive, 0=negative)
Tokens: ['this', 'is', 'terrible', '<PAD>']
--------------------------------------------------
sentence: i like this movie
Predicted label: 1 (1=postive, 0=negative)
Tokens: ['i', 'like', 'this', 'movie']
--------------------------------------------------
sentence: this is boring
Predicted label: 0 (1=postive, 0=negative)
Tokens: ['this', 'is', 'boring', '<PAD>']
--------------------------------------------------
Attention matrix example
Sentence: ['this is terrible']
Predicted Label: 0
f{'':.12}        this          is      boring       <PAD>
this              0.3352      0.1806      0.4842      0.0000
is                0.0208      0.0164      0.9628      0.0000
boring            0.4077      0.3679      0.2244      0.0000
<PAD>             0.3333

In [ ]:
import math
# 순서 정보(위치 정보) 입력
## self-attention은 단어 위치를 파악하지 못하기 떄문에, position Encoding을 통해 "어떤 단어가 앞에 있고 뒤에 있는지"
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model) #(max_len, d_model)
        # 모든 값이 0인(최대 문장 길이 x 모델 차원) 크기의 행렬을 만듦.
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) #(max_len,1)
        # 0부터 max_len까지 위치 인덱스(0,1,2,...)를 생성

        div_term = torch.exp(
            torch.arange(0,d_model,2,dtype=torch.float) * (-math.log(10000.0)/d_model)
        )
        ## 수식에서의 주기함수(sin,cos)에 들어갈 주파수 값(분모 값)을 계산

        pe[:,0::2] = torch.sin(position * div_term)
        # 짝수 인덱스에서는 인덱스(0,2,4,6...)에는 사인함수값을 채움
        pe[:,1::2] = torch.cos(position * div_term)
        # 홀수 인덱스에서는 인덱스(1,3,5,7...)에는 코사인함수값을 채움

        pe = pe.unsqueeze(0) #(1,max_len, d_model)
        # 배치 연산을 위해 차원을 확장
        self.register_buffer("pe",pe)
        # 학습되지 않는 파라미터(상수)를 설정하여 모델 저장 시 함께 저장.

    def forward(self,x):
        # x: (batch, seq_len, d_model)
        seq_len = x.size(1)
        # 입력 데이터의 문장 길이만큼 위치 정보를 가져옴
        return x + self.pe[:,:seq_len,:]
        # 원래 단어 임베딩(x)에 위치정보(pe)를 더해서 반환

- sine와 cosine 함수를 쓰는 이유?
    - 서로 다른 주기를 가진 함수들을 조합하면, 모델이 특정 위치 간의 상대적인 거리를 쉽게 학습할 수 있음.

In [ ]:
# Self-Attention Classifier
class SelfAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx, max_len):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx = pad_idx)
        self.pos_encoder = PositionalEncoding(embed_dim, max_len = max_len)
        self.W_q = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.classifier = nn.Linear(hidden_dim, num_classes)
    def forward(self,input_ids):

        # 1)padding_mask
        mask = (input_ids != self.pad_idx).unsqueeze(1) # (batch,1,seq_len)

        # embedding
        x = self.embedding(input_ids) #(batch, seq_len, embed_dim)

        #positional encoding 추가
        # self attention 코드에서 아래 코드만 추가되면
        x = self.pos_encoder(x) # (batch, seq_len, embed_dim)

        # Q,K,V
        Q = self.W_q(x) #(batch,seq_len, hidden_dim)
        K = self.W_k(x)
        V = self.W_v(x)

        # self-attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V,mask)

        # masked mean pooling
        valid_mask = (input_ids != self.pad_idx).unsqueeze(-1) #(batch, seq_len,1)
        # 패딩 토큰 위치는 0(False), 실제 단위 위치는 1(True)로 표시한 마스크를 만듦.
        attn_output = attn_output * valid_mask
        # 어텐션 결과값에 마스크를 곱해 패딩 영역의 값을 모두 0으로 초기화

        sum_output = attn_output.sum(dim=1) # (batch,hidden)
        # 패딩을 제외한 실제 단어의 벡터를 모두 더함

        valid_token_count = valid_mask.sum(dim=1).clamp(min=1) #(batch,1)
        # 각 문장별로 패딩이 아닌 실제 단어가 몇 개인지 세어줌
        # (0으로 나누기 방지를 위해 최소값 1을 설정

        pooled = sum_output / valid_token_count
        # 총합을 실제 단어 개수로 나누어, 순수하게 단어 정보만 반영된 평균 벡터를 얻음

        # Classifier
        logits = self.classifier(pooled) # (batch, num_classes)
        # 딥러닝 분류 모델의 가장 마지막 층에서 출력되는 softmax함수를 거치기 전의 가공되지 않은 예측값
        # softmax(로짓 점수)-> 모든 값이 합이 1인 확률로 변함
        return logits, attn_weights


In [ ]:
# 모델 생성
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model = SelfAttentionClassifier(
    vocab_size=len(vocab),
    embed_dim=16,
    hidden_dim=16,
    num_classes=2,
    pad_idx=pad_idx,
    max_len = max_len
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def predict_sentence(model, sentence):
    model.eval()

    ids = encode(sentence)
    input_tensor = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():
        logits, attn_weights = model(input_tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred = torch.argmax(logits, dim=1).item()

    tokens = [idx2word[idx] for idx in ids]
    return pred, probs.cpu(), attn_weights[0].cpu(), tokens


test_sentences = [
    "i love this",
    "this is terrible",
    "i like this movie",
    "this is boring",
    "this film is great",
    "i hate this"
]

print("\nPrediction results:")
for sentence in test_sentences:
    pred, probs, attn, tokens = predict_sentence(model, sentence)
    print(f"Sentence: {sentence}")
    print(f"Predicted label: {pred} (1=positive, 0=negative)")
    print(f"Probabilities : {probs.tolist()}")
    print(f"Tokens        : {tokens}")
    print("-" * 50)


Prediction results:


TypeError: linear(): argument 'input' (position 1) must be Tensor, not PositionalEncoding

In [ ]:
sentences = "this is terrible"
pred, probs, attn, tokens = predict_sentence(model, sentence)

print("Attention matrix example")
print("Sentence:", sentences)
print("Predicted Label:",pred)
print("Probabilities:", probs.tolist())
print_attention_matrix(attn,tokens)

In [27]:
# .tolist()
## 텐서나 배열을 파이썬의 기본 list로 바꾸는 것 -> 다른 라이브러리와 연동, 결과를 저장
### 모델의 출력 결과(Logits)라고 가정
logits = torch.tensor([[-0.5, 2.1], [1.5,-0.2]])

# 가장 높은 점수의 인덱스(0 또는 1)를 뽑음
pred = torch.argmax(logits, dim=1)
## dim=1: 행을 따라 왼쪽에서 오른쪽으로 이동하며 연산 / 행 내부에서 가장 큰 값의 위치를 찾으라는 명령
## [0.5, 2.1] -> 1 index 반환
## [1.5, -0.2] -> 0 index 반환
## 그래서 tensor([1,0])
print(f"Tensor 상태: {pred}") # Tensor([1,0])

# 파이썬 리스트로 변환(웹 서버로 보내거나 파일에 저장할때 필수)
pred_list = pred.tolist()
print(f"List 상태: {pred_list}") #[1,0]


Tensor 상태: tensor([1, 0])
List 상태: [1, 0]


### 3. Multi-Head Attention

In [ ]:
# Tensor-> GPU-> tolist error
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

gpu_tensor = torch.tensor([1.0,2.0,3.0]).to(device)

try:    
    bad_list = gpu_tensor.tolist()              
    ## tolist()는 내부적으로 GPU에 있는 데이터를 CPU로 복사하는 과정을 자동으로 처리해주기 때문에 에러 없이 작동
    ## numpy()는 텐서와 메모리를 공유하려고 하기 때문에 GPU 메모리에 직접 접근할 수 없기 때문에, 에러 발생
except TypeError as e:
    print(f"에러 발생!: {e}")
    # 에러 발생!: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

# 1. .cpu() -> GPU 메모리에 있는 데이터를 연산 가능한 CPU 메모리로 복사
# 2.  .detach() -> 역전파를 위한 연산 그래프의 연결을 끊음.
# 3. .tolist() -> 최종적으로 파이썬 리스트 변환.

clean_list = gpu_tensor.cpu().detach().tolist()
print(f"결과 리스트: {clean_list}")
print(f"타입 확인 : {type(clean_list)}")

# JSON 만들 때 많이 사용함

Using device: cpu


In [ ]:
class SelfAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx, max_len, num_heads):
        super().__init__()
        self.pad_idx = pad_idx
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim

        # 전체 hidden_dim을 heads로 나누어 각 head가 담당할 차원(head_dim)을 계산
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"
        self.head_dim = hidden_dim // num_heads

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(embed_dim, max_len=max_len)
        # 위치 정보를 더해주기 위한 PositionalEncoding 층을 추가함.

        self.W_q = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, hidden_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, hidden_dim, bias=False)


        self.out_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.classifier = nn.Linear(hidden_dim, num_classes)
        # 여러 헤드에서 나온 결과들을 합친 후 최종적으로 선형 변환을 해줄 층 추가.

    def forward(self, input_ids):
        """
        input_ids: (batch, seq_len)
        """

        # key padding mask for attention
        # (batch, 1, 1, seq_len)
        mask = (input_ids != self.pad_idx).unsqueeze(1).unsqueeze(2)
        # Multi-head 연산을 위해 마스크의 차원을 (batch,head,q_len, k_len) 형태로 변환.

        # embedding
        x = self.embedding(input_ids)   # (batch, seq_len, embed_dim)

        # positional encoding 추가
        x = self.pos_encoder(x)         # (batch, seq_len, embed_dim)

        # linear projections
        Q = self.W_q(x)                 # (batch, seq_len, hidden_dim)
        K = self.W_k(x)
        V = self.W_v(x)

        batch_size, seq_len, _ = Q.shape

        # split into heads
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        # (batch, num_heads, seq_len, head_dim)
        # hidden_dim을 (헤드수, 헤드별 차원)으로 쪼개고 연산을 위해 head 위치를 앞으로 옮김.

        # multi-head attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        # attn_output: (batch, num_heads, seq_len, head_dim)
        # attn_weights: (batch, num_heads, seq_len, seq_len)
        # 각 헤드가 서로 다른 부분에 집중하며 어텐션을 독립적으로 수행

        # concat heads
        attn_output = attn_output.transpose(1, 2).contiguous()
        # 흩어져 있던 헤드들을 다시 원래 순서대로 정렬하고 메모리를 연속적으로 재배치
        attn_output = attn_output.view(batch_size, seq_len, self.hidden_dim)
        # (batch, seq_len, hidden_dim)
        # 각 헤드의 결과물들을 이어 붙여(Concat) 원래의 hidden_dim 크기로 되돌림.

        # final projection
        attn_output = self.out_proj(attn_output)
        # 통합된 멀티헤드 어텐션 결과에 가중치를 곱해 최종 특징 값을 추출

        # masked mean pooling
        valid_mask = (input_ids != self.pad_idx).unsqueeze(-1)   # (batch, seq_len, 1)
        attn_output = attn_output * valid_mask

        sum_output = attn_output.sum(dim=1)                      # (batch, hidden_dim)
        valid_token_count = valid_mask.sum(dim=1).clamp(min=1)   # (batch, 1)
        pooled = sum_output / valid_token_count

        # classifier
        logits = self.classifier(pooled)

        return logits, attn_weights

In [ ]:
# 모델 생성
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model = SelfAttentionClassifier(
    vocab_size=len(vocab),
    embed_dim=16,
    hidden_dim=16,
    num_classes=2,
    pad_idx=pad_idx,
    max_len = max_len,
    num_heads=4
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def predict_sentence(model, sentence):
    model.eval()

    ids = encode(sentence)
    input_tensor = torch.tensor([ids], dtype=torch.long).to(device)

    with torch.no_grad():
        logits, attn_weights = model(input_tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred = torch.argmax(logits, dim=1).item()

    tokens = [idx2word[idx] for idx in ids]
    return pred, probs.cpu(), attn_weights[0].cpu(), tokens


test_sentences = [
    "i love this",
    "this is terrible",
    "i like this movie",
    "this is boring",
    "this film is great",
    "i hate this"
]

print("\nPrediction results:")
for sentence in test_sentences:
    pred, probs, attn, tokens = predict_sentence(model, sentence)
    print(f"Sentence: {sentence}")
    print(f"Predicted label: {pred} (1=positive, 0=negative)")
    print(f"Probabilities : {probs.tolist()}")
    print(f"Tokens        : {tokens}")
    print("-" * 50)
    # attn_weights[0] = (num_heads, seq_len,seq_len)

In [ ]:
sentences = "this is terrible"
pred, probs, attn, tokens = predict_sentence(model, sentence)

print("Attention matrices by head")
print("Sentence:", sentence)
print("predicted label:", pred)
print("Probabilties:", probs.tolist())

for h in range(attn.size(0)):
    print(f"[head {h}]")
    print_attention_matrix(attn[h], tokens)

In [ ]:
# head별 attention matrix 추가
def print_attention_matrix(attn_weights, tokens):
    print("f{'':12}", end="")
    for token in tokens:
        print(f"{token:>12}", end="")
    print()

    for i,row_token in enumerate(tokens):
        print(f"{row_token:12}", end="")
        for j in range(len(tokens)):
            print(f"{attn_weights[i,j].item():12.4f}",end="")
        print()
        
sentences = "this is terrible"
pred, probs, attn, tokens = predict_sentence(model, sentence)

print("Attention matrices by head")
print("Sentence:", sentence)
print("predicted label:", pred)
print("Probabilties:", probs.tolist())

for h in range(attn.size(0)):
    print(f"[head {h}]")
    print_attention_matrix(attn[h], tokens)